# My Private Knowledge Companion
Conversational assistant over personal notes using RAG, Chroma, and Gradio.


In [ ]:
!pip install -qqq langchain-community langchain-text-splitters langchain-openai langchain-chroma python-dotenv gradio

In [19]:
import os
from pathlib import Path
from dotenv import load_dotenv
import gradio as gr
from google.colab import userdata # Import userdata to access Colab secrets

from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

load_dotenv(override=True)

# Retrieve API key from Colab secrets
if os.getenv("OPENAI_API_KEY") is None:
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

BASE_DIR = Path().resolve()
NOTES_DIR = BASE_DIR / "knowledge_base"
CHROMA_DIR = BASE_DIR / "vector_db"

EMBEDDING_MODEL = "text-embedding-3-large"
LLM_MODEL = "gpt-4o-mini"

## Load Documents

In [20]:
def load_documents():
    loader = DirectoryLoader(
        str(NOTES_DIR),
        glob="**/*.md",
        loader_cls=TextLoader,
        loader_kwargs={"encoding": "utf-8"},
    )
    return loader.load()

documents = load_documents()
print(f"Loaded {len(documents)} documents")

Loaded 3 documents


## Split Documents

In [21]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = splitter.split_documents(documents)
print(f"Created {len(chunks)} chunks")

Created 3 chunks


## Create Vector Store

In [22]:
embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=str(CHROMA_DIR),
)

print("Vector store created")

Vector store created


## Retrieval Function

In [23]:
def retrieve_context(query, score_threshold=0.8):
    results = vectorstore.similarity_search_with_score(query, k=6)
    filtered_docs = []
    for doc, score in results:
        if score < score_threshold:
            filtered_docs.append(doc)

    if not filtered_docs:
        filtered_docs = [doc for doc, _ in results[:3]]

    return filtered_docs

## LLM Setup

In [24]:
llm = ChatOpenAI(
    model=LLM_MODEL,
    temperature=0,
    streaming=True
)

SYSTEM_PROMPT = """
You are my private knowledge companion.
Use the provided context from my notes to answer.
If the answer is not in the notes, say you don't know.
"""

## Chat Function

In [25]:
def chat_fn(message, history):
    docs = retrieve_context(message)
    context = "\n\n".join([doc.page_content for doc in docs])

    messages = [SystemMessage(content=SYSTEM_PROMPT + "\n\nContext:\n" + context)]

    for user_msg, assistant_msg in history:
        messages.append(HumanMessage(content=user_msg))
        messages.append(AIMessage(content=assistant_msg))

    messages.append(HumanMessage(content=message))

    response = ""
    for chunk in llm.stream(messages):
        if chunk.content:
            response += chunk.content
            yield response

## Gradio UI

In [ ]:
with gr.Blocks(title="My Private Knowledge Companion", theme=gr.themes.Soft()) as demo:

    gr.Markdown("# My Private Knowledge Companion Chat.")
    chatbot = gr.Chatbot(height=300)
    msg = gr.Textbox(label="Ask something about your notes")

    def respond(message, history):
        history = history + [(message, "")]
        for partial in chat_fn(message, history[:-1]):
            history[-1] = (message, partial)
            yield history, history

    msg.submit(respond, [msg, chatbot], [chatbot, chatbot])

demo.launch()